# SupportOps AI - BANKING77 Dataset Validation

## Objective

Evaluate BANKING77 as the primary intent-classification dataset for
SupportOps AI before training classical ML and Transformer models.

The validation process checks:

- Dataset structure
- Intent labels
- Missing values
- Exact duplicates
- Conflicting labels
- Train/test leakage
- Class distribution
- Text length
- Sample quality

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from datasets import load_dataset

print("Libraries imported successfully!")

Libraries imported successfully!


Download BANKING77 directly from Hugging Face

In [3]:
TRAIN_URL = (
    "https://raw.githubusercontent.com/"
    "PolyAI-LDN/task-specific-datasets/"
    "master/banking_data/train.csv"
)

TEST_URL = (
    "https://raw.githubusercontent.com/"
    "PolyAI-LDN/task-specific-datasets/"
    "master/banking_data/test.csv"
)

train_df = pd.read_csv(TRAIN_URL)
test_df = pd.read_csv(TEST_URL)

print("Training shape:", train_df.shape)
print("Testing shape:", test_df.shape)

train_df.head()

Training shape: (10003, 2)
Testing shape: (3080, 2)


,text,category
0,I am still waiting on my card?,card_arrival
1,What can I do if my card still hasn't arrived ...,card_arrival
2,I have been waiting over a week. Is the card s...,card_arrival
3,Can I track my card while it is in the process...,card_arrival
4,"How do I know if I will get my card, or if it ...",card_arrival


In [4]:
train_df = train_df.rename(
    columns={"category": "intent"}
)

test_df = test_df.rename(
    columns={"category": "intent"}
)

train_df.head()

,text,intent
0,I am still waiting on my card?,card_arrival
1,What can I do if my card still hasn't arrived ...,card_arrival
2,I have been waiting over a week. Is the card s...,card_arrival
3,Can I track my card while it is in the process...,card_arrival
4,"How do I know if I will get my card, or if it ...",card_arrival


In [5]:
intent_names = sorted(
    train_df["intent"].unique()
)

print("Number of intents:", len(intent_names))
print(intent_names[:20])

Number of intents: 77
['Refund_not_showing_up', 'activate_my_card', 'age_limit', 'apple_pay_or_google_pay', 'atm_support', 'automatic_top_up', 'balance_not_updated_after_bank_transfer', 'balance_not_updated_after_cheque_or_cash_deposit', 'beneficiary_not_allowed', 'cancel_transfer', 'card_about_to_expire', 'card_acceptance', 'card_arrival', 'card_delivery_estimate', 'card_linking', 'card_not_working', 'card_payment_fee_charged', 'card_payment_not_recognised', 'card_payment_wrong_exchange_rate', 'card_swallowed']


In [6]:
print("Training examples:", len(train_df))
print("Testing examples:", len(test_df))
print("Number of intents:", train_df["intent"].nunique())

Training examples: 10003
Testing examples: 3080
Number of intents: 77


In [7]:
print("TRAIN MISSING VALUES")
print(train_df.isnull().sum())

print("\nTEST MISSING VALUES")
print(test_df.isnull().sum())

TRAIN MISSING VALUES
text      0
intent    0
dtype: int64

TEST MISSING VALUES
text      0
intent    0
dtype: int64


In [8]:
print(
    "Exact duplicate training texts:",
    train_df.duplicated(
        subset=["text"]
    ).sum()
)

print(
    "Exact duplicate test texts:",
    test_df.duplicated(
        subset=["text"]
    ).sum()
)

Exact duplicate training texts: 0
Exact duplicate test texts: 0


In [9]:
train_label_counts = (
    train_df
    .groupby("text")["intent"]
    .nunique()
)

train_conflicts = train_label_counts[
    train_label_counts > 1
]

print(
    "Training texts with conflicting intents:",
    len(train_conflicts)
)

Training texts with conflicting intents: 0


In [10]:
train_texts = set(
    train_df["text"].str.strip()
)

test_texts = set(
    test_df["text"].str.strip()
)

overlap = train_texts.intersection(
    test_texts
)

print(
    "Exact texts appearing in both train and test:",
    len(overlap)
)

Exact texts appearing in both train and test: 6


In [11]:
overlap_examples = []

for text in sorted(overlap):
    train_labels = train_df.loc[
        train_df["text"].str.strip() == text,
        "intent"
    ].unique()

    test_labels = test_df.loc[
        test_df["text"].str.strip() == text,
        "intent"
    ].unique()

    overlap_examples.append({
        "text": text,
        "train_intent": list(train_labels),
        "test_intent": list(test_labels)
    })

overlap_df = pd.DataFrame(overlap_examples)

overlap_df

,text,train_intent,test_intent
0,At which ATMs can I use this card?,[atm_support],[atm_support]
1,How do I unblock my PIN?,[pin_blocked],[pin_blocked]
2,There are a few transaction that I don't recog...,[compromised_card],[compromised_card]
3,What businesses accept this card?,[card_acceptance],[card_acceptance]
4,Where can I use my card?,[card_acceptance],[card_acceptance]
5,Which cash machines will allow me to change my...,[change_pin],[change_pin]


Strict de-duplicated test score

In [12]:
strict_test_df = test_df[
    ~test_df["text"].str.strip().isin(train_texts)
].copy()

print("Official test size:", len(test_df))
print("Strict test size:", len(strict_test_df))

Official test size: 3080
Strict test size: 3074


In [13]:
strict_overlap = set(
    train_df["text"].str.strip()
).intersection(
    set(strict_test_df["text"].str.strip())
)

print(
    "Strict train/test exact overlap:",
    len(strict_overlap)
)

Strict train/test exact overlap: 0


In [14]:
def normalize_text(text):
    return " ".join(
        str(text)
        .lower()
        .strip()
        .split()
    )

train_normalized = set(
    train_df["text"].apply(normalize_text)
)

test_normalized = set(
    test_df["text"].apply(normalize_text)
)

normalized_overlap = train_normalized.intersection(
    test_normalized
)

print(
    "Normalized train/test overlap:",
    len(normalized_overlap)
)

Normalized train/test overlap: 7


In [15]:
def normalize_text(text):
    return " ".join(
        str(text)
        .lower()
        .strip()
        .split()
    )


# Create normalized versions
train_df["normalized_text"] = train_df["text"].apply(normalize_text)
test_df["normalized_text"] = test_df["text"].apply(normalize_text)

# All normalized texts seen during training
train_normalized_texts = set(train_df["normalized_text"])

# Create strict test set
strict_test_df = test_df[
    ~test_df["normalized_text"].isin(train_normalized_texts)
].copy()

print("Official test size:", len(test_df))
print("Strict test size:", len(strict_test_df))

Official test size: 3080
Strict test size: 3073


In [16]:
strict_overlap = set(
    train_df["normalized_text"]
).intersection(
    set(strict_test_df["normalized_text"])
)

print(
    "Normalized overlap after strict filtering:",
    len(strict_overlap)
)

Normalized overlap after strict filtering: 0


In [17]:
overlap_records = []

for normalized_text in normalized_overlap:

    train_rows = train_df[
        train_df["normalized_text"] == normalized_text
    ]

    test_rows = test_df[
        test_df["normalized_text"] == normalized_text
    ]

    overlap_records.append({
        "text": normalized_text,
        "train_intents": train_rows["intent"].unique().tolist(),
        "test_intents": test_rows["intent"].unique().tolist()
    })

pd.DataFrame(overlap_records)

,text,train_intents,test_intents
0,at which atms can i use this card?,[atm_support],[atm_support]
1,there are a few transaction that i don't recog...,[compromised_card],[compromised_card]
2,what businesses accept this card?,[card_acceptance],[card_acceptance]
3,i don't live in the uk. can i still get a card?,[country_support],[country_support]
4,how do i unblock my pin?,[pin_blocked],[pin_blocked]
5,which cash machines will allow me to change my...,[change_pin],[change_pin]
6,where can i use my card?,[card_acceptance],[card_acceptance]


In [18]:
strict_overlap = set(
    train_df["normalized_text"]
).intersection(
    set(strict_test_df["normalized_text"])
)

print(
    "Normalized overlap after strict filtering:",
    len(strict_overlap)
)

Normalized overlap after strict filtering: 0
